# 02 — Prompt Templates
Learn how to build reusable, parameterised prompts using `ChatPromptTemplate`.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


## 1. Basic ChatPromptTemplate

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define a template with named placeholders
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Answer concisely in {language}."),
    ("human", "Question: {question}")
])

# Fill the template with runtime values
prompt = template.invoke({
    "domain": "machine learning",
    "language": "English",
    "question": "What is the vanishing gradient problem?"
})

# Inspect the formatted messages
for msg in prompt.to_messages():
    print(f"[{msg.__class__.__name__}] {msg.content}\n")


## 2. Template + Model (LCEL Chain)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Pipe operator composes template → model → parser
chain = template | llm | StrOutputParser()

result = chain.invoke({
    "domain": "data structures",
    "language": "English",
    "question": "What is the difference between a stack and a queue?"
})
print(result)


## 3. Few-Shot Prompt Template

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Provide examples to guide the model's output style
examples = [
    {"input": "happy",   "output": "sad"},
    {"input": "tall",    "output": "short"},
    {"input": "fast",    "output": "slow"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai",    "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Give the antonym of every word the user provides."),
    few_shot_prompt,
    ("human", "{word}"),
])

chain = final_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()
print(chain.invoke({"word": "bright"}))


## 4. Inspecting Template Variables

In [ ]:
# List all required input variables for a template
print("Required variables:", template.input_variables)


---
**Key takeaway:** Templates are typed interfaces. `input_variables` enforces that all placeholders are filled before the chain runs.